# Project: Document Reviewer

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will build a document review agent with persistence and human-in-the-loop. The agent generates a draft, pauses for human review via `interrupt`, and revises based on feedback using `Command(resume=)`.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define the State

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
    draft: str
    feedback: str
    approved: bool

## 4. Build the Nodes

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.types import interrupt

llm = ChatOpenAI(model="gpt-4o-mini")

def generate_draft(state: State) -> dict:
    context = []
    if state.get("feedback"):
        context.append(("system", f"Previous feedback to address: {state['feedback']}"))
        context.append(("system", f"Previous draft to revise:\n{state['draft']}"))

    response = llm.invoke([
        ("system", "You are a professional writer. Write or revise a document based on the request."),
        *context,
        *state["messages"],
    ])
    return {"draft": response.content}

def human_review(state: State) -> dict:
    decision = interrupt({
        "draft": state["draft"],
        "instruction": "Review the draft. Respond with 'approve' or provide revision feedback.",
    })

    if decision.strip().lower() == "approve":
        return {"approved": True, "feedback": ""}
    return {"approved": False, "feedback": decision}

def finalize(state: State) -> dict:
    return {"messages": [("ai", f"Document approved! Final version:\n\n{state['draft']}")]}

## 5. Define Routing After Review

In [ ]:
def route_after_review(state: State) -> str:
    if state["approved"]:
        return "finalize"
    return "generate_draft"

## 6. Assemble the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

graph = StateGraph(State)
graph.add_node("generate_draft", generate_draft)
graph.add_node("human_review", human_review)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate_draft")
graph.add_edge("generate_draft", "human_review")
graph.add_conditional_edges("human_review", route_after_review, {
    "finalize": "finalize",
    "generate_draft": "generate_draft",
})
graph.add_edge("finalize", END)

checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)

## 7. Run — Generate First Draft

The graph generates a draft and pauses at the `human_review` node.

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "doc-review-1"}}

result = app.invoke(
    {"messages": [("human", "Write a product announcement for our new AI coding assistant")]},
    config=config,
)
print("Draft generated. Waiting for review...")

state = app.get_state(config)
print(f"\nDraft:\n{state.values['draft']}")

## 8. Resume — Request Revision

Provide feedback to trigger a revision loop.

In [ ]:
result = app.invoke(
    Command(resume="Make it more concise and add a call-to-action at the end"),
    config=config,
)
print("Revision submitted. Waiting for review...")

state = app.get_state(config)
print(f"\nRevised draft:\n{state.values['draft']}")

## 9. Resume — Approve Final Version

In [ ]:
result = app.invoke(Command(resume="approve"), config=config)
print(result["messages"][-1].content)

## 10. Inspect the Review History

Walk through the checkpoint history to see each step.

In [ ]:
for snapshot in app.get_state_history(config):
    print(f"Step {snapshot.metadata['step']}: next={snapshot.next}")
    if snapshot.values.get("draft"):
        print(f"  Draft length: {len(snapshot.values['draft'])} chars")
    if snapshot.values.get("feedback"):
        print(f"  Feedback: {snapshot.values['feedback']}")

## What You Learned

1. **`interrupt(value)`** pauses execution and surfaces data for human review
2. **`Command(resume=value)`** continues from the exact interrupt point
3. A **checkpointer is required** for human-in-the-loop workflows
4. **Conditional edges** after review create a revision loop until approval
5. This pattern applies to any workflow needing **human oversight** — content review, code review, approval chains